In [ ]:
from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!pip install timm grad-cam groq

Saving kaggle.json to kaggle.json
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 66.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 14.0 MB/s eta 0:00:00
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44286 sha256=df0f1b25d1fa893e9fdb5c3b0e477202fef7cfa60400d4d9d4d70a6947d5f2fa
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam


In [ ]:
!kaggle datasets download -d mohamedhanyyy/chest-ctscan-images
!unzip chest-ctscan-images.zip -d data

Dataset URL: https://www.kaggle.com/datasets/mohamedhanyyy/chest-ctscan-images
License(s): ODbL-1.0
100% 119M/119M [00:01<00:00, 89.7MB/s]

Archive:  chest-ctscan-images.zip
  inflating: data/Data/test/adenocarcinoma/000108 (3).png  
  inflating: data/Data/test/adenocarcinoma/000109 (2).png  
  inflating: data/Data/test/adenocarcinoma/000109 (4).png  
  inflating: data/Data/test/adenocarcinoma/000109 (5).png  
  inflating: data/Data/test/adenocarcinoma/000112 (2).png  
  inflating: data/Data/test/adenocarcinoma/000113 (7).png  
  inflating: data/Data/test/adenocarcinoma/000114 (5).png  
  inflating: data/Data/test/adenocarcinoma/000114.png  
  inflating: data/Data/test/adenocarcinoma/000115 (4).png  
  inflating: data/Data/test/adenocarcinoma/000115 (8).png  
  inflating: data/Data/test/adenocarcinoma/000115.png  
  inflating: data/Data/test/adenocarcinoma/000116 (5).png  
  inflating: data/Data/test/adenocarcinoma/000116 (7).png  
  inflating: data/Data/test/adenocarcinoma/000116 (9).

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

TRAIN_PATH = "data/Data/train"
VAL_PATH   = "data/Data/valid"
TEST_PATH  = "data/Data/test"

IMG_SIZE = 224
BATCH_SIZE = 32

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_data = datasets.ImageFolder(TRAIN_PATH, transform=transform)
val_data   = datasets.ImageFolder(VAL_PATH, transform=transform)
test_data  = datasets.ImageFolder(TEST_PATH, transform=transform)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_data, batch_size=BATCH_SIZE)

print("Classes:", train_data.classes)

Classes: ['adenocarcinoma_left.lower.lobe_T2_N0_M0_Ib', 'large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa', 'normal', 'squamous.cell.carcinoma_left.hilum_T1_N2_M0_IIIa']


In [ ]:
import torch
import torch.nn as nn
import timm

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = timm.create_model('efficientnet_b0', pretrained=True)

num_classes = len(train_data.classes)
model.classifier = nn.Linear(model.classifier.in_features, num_classes)

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Acc: {100*correct/total:.2f}%")

Epoch 1 | Loss: 1.0598 | Acc: 62.48%
Epoch 2 | Loss: 0.6094 | Acc: 83.52%
Epoch 3 | Loss: 0.2984 | Acc: 95.60%
Epoch 4 | Loss: 0.1466 | Acc: 98.21%
Epoch 5 | Loss: 0.0755 | Acc: 98.86%


In [ ]:
from sklearn.metrics import classification_report

model.eval()

y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        y_true.extend(labels.numpy())
        y_pred.extend(preds)

print(classification_report(y_true, y_pred, target_names=train_data.classes))

                                                  precision    recall  f1-score   support

      adenocarcinoma_left.lower.lobe_T2_N0_M0_Ib       0.89      0.93      0.91       120
   large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa       0.88      0.82      0.85        51
                                          normal       1.00      0.98      0.99        54
squamous.cell.carcinoma_left.hilum_T1_N2_M0_IIIa       0.95      0.93      0.94        90

                                        accuracy                           0.92       315
                                       macro avg       0.93      0.92      0.92       315
                                    weighted avg       0.92      0.92      0.92       315



In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import numpy as np
import matplotlib.pyplot as plt

target_layer = model.conv_head

cam = GradCAM(model=model, target_layers=[target_layer])

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import numpy as np
import matplotlib.pyplot as plt

target_layer = model.conv_head

cam = GradCAM(model=model, target_layers=[target_layer])

In [ ]:
def interpret_cam(cam_map):
    high = np.mean(cam_map > 0.6)
    mid = np.mean((cam_map > 0.3) & (cam_map <= 0.6))

    return f"""
    The model focuses strongly on {high*100:.2f}% tumor regions,
    and moderately on {mid*100:.2f}% areas.
    This indicates detection of abnormal lung tissue patterns.
    """

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving 206.png to 206.png


In [ ]:
image_path = "206.png"

In [ ]:
from PIL import Image
image = Image.open(image_path).convert('RGB')
input_tensor = transform(image).unsqueeze(0).to(device)

In [ ]:
import torch.nn.functional as F
model.eval()
with torch.no_grad():
    outputs = model(input_tensor)
    # Apply softmax to get probabilities for all classes
    probabilities = F.softmax(outputs, dim=1)[0]
    # Get the top prediction and its confidence score
    conf, predicted = torch.max(probabilities, 0)

# 4. Print results and all confidence levels
class_names = train_data.classes
print(f"--- Results for: {image_path} ---")
print(f"Final Prediction: {class_names[predicted.item()]} ({conf.item()*100:.2f}%)\n")

print("Confidence Levels:")
for i, class_name in enumerate(class_names):
    score = probabilities[i].item() * 100
    print(f"- {class_name}: {score:.2f}%")

--- Results for: 206.png ---
Final Prediction: squamous.cell.carcinoma_left.hilum_T1_N2_M0_IIIa (94.70%)

Confidence Levels:
- adenocarcinoma_left.lower.lobe_T2_N0_M0_Ib: 4.51%
- large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa: 0.35%
- normal: 0.45%
- squamous.cell.carcinoma_left.hilum_T1_N2_M0_IIIa: 94.70%


In [ ]:
class_names = train_data.classes
print("Prediction:", class_names[predicted.item()])

Prediction: squamous.cell.carcinoma_left.hilum_T1_N2_M0_IIIa


In [ ]:
import torch
from google.colab import files

# 1. Define the filename
model_path = "lung_cancer_model.pth"

# 2. Save the model state dictionary
torch.save(model.state_dict(), model_path)

# 3. Download to your computer
files.download(model_path)
print(f"Model saved and download triggered: {model_path}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Model saved and download triggered: lung_cancer_model.pth


In [ ]:
import pickle
from google.colab import files

# 1. Define the filename
pkl_path = "lung_cancer_model.pkl"

# 2. Save using pickle
with open(pkl_path, 'wb') as f:
    pickle.dump(model, f)

# 3. Download to your computer
files.download(pkl_path)
print(f"Pickle model saved and download triggered: {pkl_path}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Pickle model saved and download triggered: lung_cancer_model.pkl
